In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:


input_path = Path("derived") / "dataset.xlsx"  # Path to the input workbook.
output_dir = Path("STY_results")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
point_id = "Data point number"  # Unique experimental-record identifier column.
data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.

sacu_provenance = pd.read_excel(  # Source and measurement method for Cu active surface area.
    "SI-dataset.xlsx", sheet_name="data", engine="openpyxl",
    usecols=["Number", "SACu Source", "N2O Method"],
).rename(columns={"Number": point_id})
sacu_records = data.merge(sacu_provenance, on=point_id, how="left", validate="one_to_one")  # Screened records joined to their Cu surface-area provenance.
if sacu_records[["SACu Source", "N2O Method"]].isna().any().any():
    raise ValueError("Check the Cu surface-area provenance for the screened records.")

support_slots = ["Support1", "Support2", "Support3"]  # Support positions in the source data.
zno_loadings = pd.DataFrame({  # ZnO loading by support position.
    slot: pd.to_numeric(sacu_records[f"wt% {slot}"], errors="coerce").where(
        sacu_records[slot].astype("string").str.strip().str.casefold().eq("zno").fillna(False), 0
    )
    for slot in support_slots  # Slot.
})
sacu_records["ZnO wt%"] = zno_loadings.sum(axis=1, skipna=False)
measurements = ["ZnO wt%", "wt% Cu", "SACu"]  
sacu_records[measurements] = sacu_records[measurements].apply(pd.to_numeric, errors="coerce")
eligible = (  # Mask for records that satisfy the measurement and composition criteria.
    np.isfinite(sacu_records[measurements]).all(axis=1)
    & sacu_records[measurements].gt(0).all(axis=1)
    & sacu_records["SACu Source"].str.strip().str.casefold().eq("reported")
    & sacu_records["N2O Method"].str.contains(r"N[2₂]O", case=False, na=False)
)
sacu_points = sacu_records.loc[eligible].copy()  # Records with eligible source-reported N2O measurements.
catalyst_key = ["Reference number", "Catalyst number"]  
if sacu_points.empty or sacu_points[catalyst_key].isna().any().any():
    raise ValueError("Eligible measurements need a publication and catalyst identifier.")
if sacu_points.groupby(catalyst_key)[measurements].nunique().gt(1).any().any():
    raise ValueError("Resolve conflicting composition or surface-area values for repeated catalysts.")

sacu_catalysts = sacu_points.drop_duplicates(catalyst_key).copy()  # Unique catalysts with eligible measurements.
sacu_catalysts["Zn/Cu molar ratio"] = (sacu_catalysts["ZnO wt%"] / 81.379) / (sacu_catalysts["wt% Cu"] / 63.546)
ratio_column = "Zn/Cu molar ratio"  # Ratio column.
ratio_bins = pd.cut(sacu_catalysts[ratio_column], [0, 0.25, 0.5, 1, 2, 5, np.inf])  # Intervals used to group Zn/Cu molar ratios.
ratio_summary = sacu_catalysts.groupby(ratio_bins, observed=True).agg(  # Median and interquartile range of Cu surface area by ratio interval.
    catalysts=("SACu", "size"), ratio=(ratio_column, "median"),
    median=("SACu", "median"), q1=("SACu", lambda values: values.quantile(0.25)),
    q3=("SACu", lambda values: values.quantile(0.75)),
)
print(f"Retained N2O records: {len(sacu_points)} | Catalysts: {len(sacu_catalysts)} | "
      f"Publications: {sacu_catalysts['Reference number'].nunique()}")
display(ratio_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True, layout="constrained")  # Matplotlib figure; Plot axes arranged by panel.
for ax, column, x_label, label in zip(  # Plot axes; Current column name; X label; Label.
    axes, ["ZnO wt%", ratio_column], ["ZnO loading (wt%)", "Zn/Cu molar ratio"], ["(a)", "(b)"]
):
    ax.scatter(sacu_catalysts[column], sacu_catalysts["SACu"], s=19,
               color="#0072B2", alpha=0.46, linewidths=0, label="Individual catalyst")
    ax.set_xlabel(x_label)
    ax.set_ylim(bottom=0)
    ax.tick_params(direction="out")
    ax.text(0.015, 1.045, label, transform=ax.transAxes, fontsize=13, fontweight="bold")

axes[0].set_ylabel(r"$SA_{\mathrm{Cu}}$ (m$^2$ g$_{\mathrm{cat}}^{-1}$)")
axes[0].set_xlim(left=0)
axes[0].legend(loc="upper right", frameon=False, fontsize=8)
axes[1].set_xscale("log")
axes[1].errorbar(
    ratio_summary["ratio"], ratio_summary["median"],
    yerr=[ratio_summary["median"] - ratio_summary["q1"], ratio_summary["q3"] - ratio_summary["median"]],
    fmt="D", color="#B94D00", markersize=4.5, capsize=3, linewidth=1.3,
    label="Bin median and interquartile range",
)
axes[1].legend(loc="upper right", frameon=False, fontsize=8)
fig.savefig(output_dir / "ZnO_Cu_ratio_SACu.png", dpi=300, bbox_inches="tight")  # Save Fig. S5a (ZnO loading) and Fig. S5b (Zn/Cu molar ratio) as a PNG.
fig.savefig(output_dir / "ZnO_Cu_ratio_SACu.tif", dpi=600, bbox_inches="tight",  # Save Fig. S5a (ZnO loading) and Fig. S5b (Zn/Cu molar ratio) as a TIFF.
            pil_kwargs={"compression": "tiff_lzw"})
plt.show()
plt.close(fig)